In [ ]:
# -*- coding: utf-8 -*-
"""01_adquisicion_cruce_diccionario.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1ubiyN8KarQ5x-hw8gCkZh_p2UPiTEUZA

Proyecto de Minería de Datos

## Desempeño Educativo en las Pruebas Saber 11

**Integrantes:** Juan David Romero, Cesar Toro y Oscar Colorado  
**Programa:** Ingeniería de Software  
**Asignatura:** Minería de Datos  
**Corte:** 1  Adquisición, cruce y diccionario de datos

### Pregunta de decisión
> ¿Qué características territoriales e institucionales se relacionan con mejores o peores desempeños en las pruebas Saber 11 en Colombia, para identificar dónde deberían priorizarse acciones de mejoramiento educativo?

Este notebook implementa la etapa de **adquisición e integración** con los archivos disponibles para el proyecto. La ficha plantea ICFES como fuente de resultados Saber 11 y DANE-EDUC como fuente de características educativas. En esta entrega, el archivo ICFES suministrado contiene 63 instituciones de municipios de Risaralda y categorías anuales de desempeño 2014–2024. La segunda fuente utilizada en el cruce es la variable de identificación municipal de DANE EDUC 2024, que permite homologar los nombres de municipio y obtener su código DANE.

**Importante:** el archivo DANE disponible para este notebook es un catálogo/tabla de identificación municipal, no la base completa de matrícula, docentes o TIC. Por tanto, el cruce demuestra la integración por identificador territorial, pero todavía no incorpora esas variables educativas de DANE. Esto se deja explícito para no inventar datos.

## 1. Librerías y rutas

Se utilizará `pandas` para lectura, exploración, limpieza y cruce de las fuentes.
"""

import pandas as pd
import numpy as np
import unicodedata
import re
from pathlib import Path

BASE = Path("..")
ICFES_PATH = BASE / "data" / "raw" / "Pruebas_ICFES_20260929.csv"
DANE_PATH = BASE / "data" / "raw" / "dane_municipios_2024.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

"""## 2. Fuente 1 — ICFES / archivo de resultados entregado

La actividad exige que las fuentes se lean correctamente con pandas y que se verifiquen mediante `.head()` y `.shape()`.

"""

df_icfes = pd.read_csv(ICFES_PATH, low_memory=False)

print("Dimensiones:", df_icfes.shape)
display(df_icfes.head())

"""### Tipos de datos y estructura

El archivo tiene 63 registros y 13 columnas originales: municipio, institución educativa y una columna de desempeño para cada año de 2014 a 2024.

"""

print(df_icfes.dtypes)
print("\nNúmero de columnas:", len(df_icfes.columns))
print("Número de registros:", len(df_icfes))

"""## 3. Calidad inicial de la fuente ICFES

Se revisan valores faltantes y duplicados antes de integrar.

"""

print("Duplicados completos:", df_icfes.duplicated().sum())
print("\nValores faltantes por columna:")
display(df_icfes.isna().sum().to_frame("faltantes"))

print("\nMunicipios encontrados:")
display(df_icfes["MUNICIPIO"].value_counts().to_frame("instituciones"))

"""### Valores de desempeño

Se conservan las categorías exactamente como aparecen en la fuente. Se detecta `Nr` en 2017, además de `NR`, por lo que en la limpieza se homologará `Nr` a `NR`. No se asigna una puntuación numérica porque el archivo entregado no contiene en sus columnas una definición explícita de equivalencia numérica para estas categorías.

"""

year_cols = [c for c in df_icfes.columns if c.startswith("AÑO")]

for col in year_cols:
    print(f"{col}: {sorted(df_icfes[col].dropna().astype(str).unique().tolist())}")

"""## 4. Limpieza y homologación de nombres

Para cruzar fuentes territoriales, se crea una clave normalizada:
- elimina espacios sobrantes;
- pasa a minúsculas;
- elimina diferencias de tildes;
- permite homologar variantes de escritura.

Además, `Santa Rosa` se homologa a `Santa Rosa de Cabal`, que es el nombre utilizado por DANE para el municipio.

"""

def normalizar_texto(valor):
    if pd.isna(valor):
        return valor
    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor).encode("ascii", "ignore").decode("ascii")
    valor = re.sub(r"\s+", " ", valor)
    return valor

df_icfes["MUNICIPIO_KEY"] = df_icfes["MUNICIPIO"].map(normalizar_texto)

# Homologación documentada de la abreviatura presente en el archivo entregado.
df_icfes["MUNICIPIO_KEY"] = df_icfes["MUNICIPIO_KEY"].replace({
    "santa rosa": "santa rosa de cabal"
})

# Homologación de la categoría escrita como "Nr".
for col in year_cols:
    df_icfes[col] = df_icfes[col].replace({"Nr": "NR"})

display(df_icfes[["MUNICIPIO", "MUNICIPIO_KEY"]].drop_duplicates().sort_values("MUNICIPIO"))

"""## 5. Fuente 2  DANE Educación Formal (EDUC) 2024

Se utiliza la tabla de identificación municipal de la **Carátula única de la sede educativa** de EDUC 2024. Contiene el código interno del municipio/distrito y su nombre. Esto permite realizar un cruce territorial reproducible.

El archivo local `dane_municipios_2024.csv` fue preparado a partir de esa fuente oficial y contiene únicamente los municipios presentes en el archivo ICFES entregado.

"""

df_dane = pd.read_csv(DANE_PATH, dtype={"CODIGO_DANE_MUNICIPIO": str})

df_dane["MUNICIPIO_KEY"] = df_dane["MUNICIPIO_DANE"].map(normalizar_texto)

print("Dimensiones DANE:", df_dane.shape)
display(df_dane.head())

"""## 6. Verificación de la llave de cruce

La llave elegida es `MUNICIPIO_KEY`, construida a partir del nombre del municipio. Antes del `merge` se comprueba que la tabla DANE tenga un único registro por municipio.

"""

print("Municipios únicos en ICFES:", df_icfes["MUNICIPIO_KEY"].nunique())
print("Municipios únicos en DANE:", df_dane["MUNICIPIO_KEY"].nunique())

print("\nDuplicados de la llave en DANE:")
display(df_dane[df_dane["MUNICIPIO_KEY"].duplicated(keep=False)])

assert not df_dane["MUNICIPIO_KEY"].duplicated().any(), "La llave DANE debe ser única para un merge many-to-one."

"""## 7. Cruce de fuentes con `pd.merge`

La rúbrica exige verificar el número de filas antes y después. Se utiliza `validate="many_to_one"` porque varias instituciones pueden pertenecer al mismo municipio, mientras que el catálogo municipal debe tener un solo registro por municipio.

"""

filas_antes = len(df_icfes)

df_cruzado = df_icfes.merge(
    df_dane,
    on="MUNICIPIO_KEY",
    how="left",
    validate="many_to_one",
    indicator=True
)

filas_despues = len(df_cruzado)

print("Filas antes del cruce:", filas_antes)
print("Filas después del cruce:", filas_despues)
print("\nResultado del cruce:")
display(df_cruzado["_merge"].value_counts())

assert filas_antes == filas_despues, "El cruce no debe duplicar registros de instituciones."
assert (df_cruzado["_merge"] == "both").all(), "Hay municipios ICFES sin correspondencia DANE."

"""### Resultado del cruce

El cruce conserva los **63 registros** de instituciones y logra correspondencia para los 12 municipios presentes. No se generan filas adicionales, lo que confirma que la relación utilizada es de muchas instituciones hacia un municipio.

"""

display(
    df_cruzado[
        ["MUNICIPIO", "MUNICIPIO_DANE", "CODIGO_DANE_MUNICIPIO", "DEPARTAMENTO", "_merge"]
    ].drop_duplicates().sort_values("MUNICIPIO")
)

"""## 8. Exploración de los desempeños 2021–2024

La ficha del proyecto propone trabajar 2021–2025 sujeto a disponibilidad y compatibilidad. El archivo entregado llega hasta 2024, por lo que aquí se analizan 2021–2024 y se deja constancia de que **2025 no está disponible en esta fuente**.

"""

years_available = [c for c in year_cols if c in ["AÑO 2021", "AÑO 2022", "AÑO 2023", "AÑO 2024"]]

frecuencias = {}
for col in years_available:
    frecuencias[col] = df_cruzado[col].value_counts(dropna=False)

frecuencias_df = pd.DataFrame(frecuencias).fillna(0).astype(int)
display(frecuencias_df)

"""## 9. Diccionario de datos

El diccionario cubre las columnas principales del dataset cruzado, indicando tipo, significado y fuente.

"""

diccionario = pd.DataFrame([
    ["MUNICIPIO", str(df_cruzado["MUNICIPIO"].dtype), "Municipio reportado en la fuente entregada.", "ICFES / CSV entregado"],
    ["INSTITUCION EDUCATIVA", str(df_cruzado["INSTITUCION EDUCATIVA"].dtype), "Nombre de la institución educativa.", "ICFES / CSV entregado"],
    ["AÑO 2014–AÑO 2024", "object", "Categoría anual de desempeño reportada; se conservan A+, A, B, C, D y NR.", "ICFES / CSV entregado"],
    ["MUNICIPIO_KEY", str(df_cruzado["MUNICIPIO_KEY"].dtype), "Clave normalizada usada para integrar las fuentes.", "Derivada del proceso de limpieza"],
    ["CODIGO_DANE_MUNICIPIO", str(df_cruzado["CODIGO_DANE_MUNICIPIO"].dtype), "Código interno del municipio o distrito.", "DANE EDUC 2024"],
    ["MUNICIPIO_DANE", str(df_cruzado["MUNICIPIO_DANE"].dtype), "Nombre del municipio según DANE.", "DANE EDUC 2024"],
    ["DEPARTAMENTO", str(df_cruzado["DEPARTAMENTO"].dtype), "Departamento correspondiente a los municipios incluidos.", "DANE / contexto del proyecto"],
    ["_merge", str(df_cruzado["_merge"].dtype), "Indicador del resultado del merge de pandas.", "Derivada del proceso de integración"],
], columns=["columna", "tipo_dato", "significado", "fuente"])

display(diccionario)

"""## 10. Exportación del dataset integrado

Se guarda una versión procesada para que el resultado del cruce sea reproducible desde el repositorio.

"""

OUTPUT_PATH = BASE / "data" / "processed" / "saber11_dane_merged.csv"

df_cruzado.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Archivo generado: {OUTPUT_PATH}")
print("Dimensiones finales:", df_cruzado.shape)

"""## 11. Riesgos y limitaciones identificados

1. **Cobertura:** el archivo ICFES entregado contiene 63 instituciones de 12 municipios de Risaralda, no una cobertura nacional.
2. **Periodo:** se dispone de 2014–2024; no hay columna 2025 en el archivo entregado.
3. **Naturaleza de la variable:** los resultados están expresados como categorías (`A+`, `A`, `B`, `C`, `D`, `NR`), no como puntajes numéricos por área.
4. **Fuente DANE:** en esta entrega solo se incorpora el identificador municipal de EDUC 2024. No se han añadido todavía matrícula, docentes, zona o TIC porque esas variables no están presentes en el archivo DANE suministrado para esta ejecución.
5. **Homologación:** `Santa Rosa` del archivo ICFES se homologó con `Santa Rosa de Cabal` del DANE.
6. **Interpretación:** una asociación o diferencia observada no debe interpretarse como causalidad. La propia ficha del proyecto establece esta precaución.

## 12. Conclusión de la etapa de adquisición

La etapa de adquisición cumple con la lectura de la fuente entregada, las verificaciones de estructura y calidad, la homologación territorial, el cruce mediante una columna común y la construcción del diccionario de datos.

**Resultado técnico del cruce:** 63 filas antes → 63 filas después; 63 correspondencias (`both`); 0 registros sin correspondencia y 0 duplicaciones generadas por el merge.

Para una siguiente versión del proyecto, se debe incorporar la base DANE EDUC con variables de matrícula, docentes, zona y/o TIC para que el cruce no sea únicamente de identificación territorial y permita responder con mayor profundidad la pregunta de decisión planteada en la ficha.
"""